# 🎧 SpatialNet — HRTF-Net Training Pipeline
### Samsung ennovateX AX Hackathon 2026 | PS08 | SoundScape AI

**What this notebook does:**
1. Installs all dependencies
2. Downloads real SONICOM HRTF data (CC-BY 4.0)
3. Trains `SpatialNet` (coordinate → binaural FIR filter)
4. Exports to ONNX FP32
5. Quantizes to INT8
6. Benchmarks 1000 inference runs → KPI report
7. Downloads all artifacts

> **Runtime:** Use `Runtime → Change runtime type → T4 GPU` for fastest training.
> Training on CPU also works fine (~30 seconds for 15 epochs on this tiny network).

---
## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# torch/torchaudio are pre-installed on Colab, but we pin versions for safety

import subprocess, sys

packages = [
    'python-sofa',
    'requests',
    'onnx',
    'onnxruntime',
    'numpy',
    'matplotlib',
    'scipy'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed.')

---
## Cell 2 — Imports & Device Check

In [ ]:
import os
import json
import time
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import sofa
import onnx
import onnxruntime as ort

# Device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'ONNX version    : {onnx.__version__}')
print(f'ORT version     : {ort.__version__}')
print(f'Device          : {device}')

if device.type == 'cuda':
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  Running on CPU — still fast enough for this tiny network.')

---
## Cell 3 — Download SONICOM Dataset (Subject P0001)

SONICOM HRTF Dataset — License: **CC-BY 4.0** ✅  
Source: https://www.sonicom.eu/

In [ ]:
def download_sonicom_sample(dest_folder='data/hrtf'):
    os.makedirs(dest_folder, exist_ok=True)
    dest_path = os.path.join(dest_folder, 'P0001.sofa')

    # Primary URL (Zenodo)
    urls = [
        'https://zenodo.org/record/5661001/files/P0001_FreeField_Clean_44100.sofa?download=1',
        'https://zenodo.org/record/5661001/files/P0001_FreeField.sofa',
    ]

    if os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / 1024**2
        print(f'✅ Already downloaded: {dest_path} ({size_mb:.1f} MB)')
        return dest_path

    for url in urls:
        print(f'📥 Trying: {url}')
        try:
            r = requests.get(url, stream=True, timeout=60)
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            downloaded = 0
            with open(dest_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        pct = downloaded / total * 100
                        print(f'   {pct:.0f}%', end='\r')
            print(f'\n✅ Download complete: {os.path.getsize(dest_path)/1024**2:.1f} MB')
            return dest_path
        except Exception as e:
            print(f'   ❌ Failed: {e}')
            if os.path.exists(dest_path):
                os.remove(dest_path)

    raise RuntimeError('Could not download SONICOM data. Check your internet connection or try a VPN.')

sofa_path = download_sonicom_sample()
print(f'SOFA file: {sofa_path}')

---
## Cell 4 — SOFA Dataset Loader

In [ ]:
class SONICOMDataset(Dataset):
    """
    Loads HRTF impulse responses from a SOFA file.
    Each sample: (source_position [3], HRIR_left [filter_length], HRIR_right [filter_length])
    """
    def __init__(self, sofa_path, filter_length=128):
        hrir_file = sofa.Database.open(sofa_path)
        self.source_pos = np.array(hrir_file.Variables['SourcePosition'][:])
        self.data       = np.array(hrir_file.Variables['Data.IR'][:])
        self.filter_length = filter_length
        self.num_samples   = self.source_pos.shape[0]

        # Normalize source positions (azimuth 0-360, elevation -90-90, distance)
        self.pos_min = self.source_pos.min(axis=0)
        self.pos_max = self.source_pos.max(axis=0)
        self.pos_range = np.where(self.pos_max - self.pos_min == 0, 1.0, self.pos_max - self.pos_min)

        print(f'✅ SOFA loaded: {self.num_samples} positions')
        print(f'   Azimuth range   : {self.source_pos[:, 0].min():.0f}° – {self.source_pos[:, 0].max():.0f}°')
        print(f'   Elevation range : {self.source_pos[:, 1].min():.0f}° – {self.source_pos[:, 1].max():.0f}°')
        print(f'   HRIR length used: {filter_length} taps')

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Normalize position to [0, 1]
        raw_pos = self.source_pos[idx]
        norm_pos = (raw_pos - self.pos_min) / self.pos_range
        pos  = torch.tensor(norm_pos, dtype=torch.float32)
        ir_l = torch.tensor(self.data[idx, 0, :self.filter_length], dtype=torch.float32)
        ir_r = torch.tensor(self.data[idx, 1, :self.filter_length], dtype=torch.float32)
        return pos, ir_l, ir_r


FILTER_LENGTH = 128
dataset    = SONICOMDataset(sofa_path, filter_length=FILTER_LENGTH)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

print(f'\nDataLoader ready: {len(dataset)} samples, {len(dataloader)} batches/epoch')

---
## Cell 5 — Model Definition (SpatialNet / HRTF-Net)

In [ ]:
class SpatialNet(nn.Module):
    """
    HRTF-Net: Maps 3D source coordinates → binaural FIR filter pair (left + right ear).
    Architecture: Linear(3→64→128→256) + dual output heads
    Target size: <8MB | Target latency: <20ms
    """
    def __init__(self, filter_length=128):
        super(SpatialNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU()
        )
        self.head_left  = nn.Linear(256, filter_length)
        self.head_right = nn.Linear(256, filter_length)

    def forward(self, coords):
        features = self.network(coords)
        return self.head_left(features), self.head_right(features)


model = SpatialNet(filter_length=FILTER_LENGTH).to(device)

# Count parameters
total_params  = sum(p.numel() for p in model.parameters())
size_bytes    = sum(p.numel() * p.element_size() for p in model.parameters())

print('Model Architecture:')
print(model)
print(f'\nTotal parameters : {total_params:,}')
print(f'In-memory size   : {size_bytes / 1024:.1f} KB ({size_bytes / 1024**2:.3f} MB)')
print(f'KPI <8MB         : {"✅ PASS" if size_bytes / 1024**2 < 8 else "❌ FAIL"}')

---
## Cell 6 — Training Loop

In [ ]:
EPOCHS    = 15
LR        = 0.001

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

loss_history = []

print(f'Starting Training on {device} for {EPOCHS} epochs...\n')
train_start = time.time()

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for pos, tl, tr in dataloader:
        pos, tl, tr = pos.to(device), tl.to(device), tr.to(device)

        optimizer.zero_grad()
        pl, pr = model(pos)
        loss = criterion(pl, tl) + criterion(pr, tr)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(dataloader)
    loss_history.append(avg_loss)

    marker = ' ◀' if (epoch + 1) % 5 == 0 else ''
    print(f'  Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.6f}{marker}')

elapsed = time.time() - train_start
print(f'\n✅ Training complete in {elapsed:.1f}s ({elapsed/EPOCHS:.1f}s/epoch)')

---
## Cell 7 — Plot Training Loss

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(range(1, EPOCHS + 1), loss_history, 'o-', color='#00BFA5', linewidth=2,
        markersize=5, label='Training Loss (L + R MSE)')
ax.fill_between(range(1, EPOCHS + 1), loss_history, alpha=0.15, color='#00BFA5')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('SpatialNet (HRTF-Net) Training Loss — PS08 SoundScape AI', fontsize=13, fontweight='bold')
ax.set_xticks(range(1, EPOCHS + 1))
ax.grid(True, alpha=0.3)
ax.legend()

# Annotate final loss
ax.annotate(f'Final: {loss_history[-1]:.6f}',
            xy=(EPOCHS, loss_history[-1]),
            xytext=(EPOCHS - 3, loss_history[-1] + (loss_history[0] - loss_history[-1]) * 0.1),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='#333')

plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: training_loss.png')

---
## Cell 8 — Save PyTorch Weights

In [ ]:
# Save the trained model weights
torch.save(model.state_dict(), 'spatialnet.pth')
pth_size = os.path.getsize('spatialnet.pth') / 1024**2

print(f'✅ Weights saved : spatialnet.pth ({pth_size:.3f} MB)')

# Quick sanity-check inference
model.eval()
with torch.no_grad():
    test_pos = torch.tensor([[0.0, 0.0, 1.0]]).to(device)   # front, ear level, 1m
    l, r = model(test_pos)
    print(f'Test inference   : left_filter shape={tuple(l.shape)}, right_filter shape={tuple(r.shape)}')
    print(f'Output range     : [{l.min().item():.4f}, {l.max().item():.4f}]')

---
## Cell 9 — Export to ONNX (FP32)

In [ ]:
# Load fresh copy of model (ensures clean export)
model_export = SpatialNet(filter_length=FILTER_LENGTH)
model_export.load_state_dict(torch.load('spatialnet.pth', map_location='cpu'))
model_export.eval()

dummy_input = torch.randn(1, 3)   # batch=1, coords=(azimuth, elevation, distance)

torch.onnx.export(
    model_export,
    dummy_input,
    'spatialnet.onnx',
    input_names=['position'],
    output_names=['hrtf_left', 'hrtf_right'],
    dynamic_axes={'position': {0: 'batch'}},
    opset_version=17,
    verbose=False
)

# Validate ONNX graph
onnx_model = onnx.load('spatialnet.onnx')
onnx.checker.check_model(onnx_model)

fp32_size = os.path.getsize('spatialnet.onnx') / 1024**2
print(f'✅ ONNX export   : spatialnet.onnx')
print(f'   Graph valid   : ✅')
print(f'   File size     : {fp32_size:.3f} MB')
print(f'   KPI <50MB     : {"✅ PASS" if fp32_size < 50 else "❌ FAIL"}')

---
## Cell 10 — ONNX Runtime Latency Benchmark (FP32)

In [ ]:
def benchmark_onnx(model_path, n_warmup=20, n_runs=1000):
    """Benchmark ONNX model inference latency."""
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    dummy   = np.random.randn(1, 3).astype(np.float32)

    # Warmup
    for _ in range(n_warmup):
        session.run(None, {'position': dummy})

    # Benchmark
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(None, {'position': dummy})
        latencies.append((time.perf_counter() - t0) * 1000)

    return np.array(latencies)


print('Running 1000 inference benchmarks (FP32)...')
lats_fp32 = benchmark_onnx('spatialnet.onnx')

print('\n=== FP32 ONNX KPI REPORT ===')
print(f'  Mean latency : {lats_fp32.mean():.4f} ms')
print(f'  P50  latency : {np.percentile(lats_fp32, 50):.4f} ms')
print(f'  P95  latency : {np.percentile(lats_fp32, 95):.4f} ms')
print(f'  P99  latency : {np.percentile(lats_fp32, 99):.4f} ms')
print(f'  Max  latency : {lats_fp32.max():.4f} ms')
print(f'  KPI <20ms    : {"✅ PASS" if lats_fp32.mean() < 20 else "❌ FAIL"}')

fp32_size = os.path.getsize('spatialnet.onnx') / 1024**2
print(f'\n  Model size   : {fp32_size:.3f} MB')
print(f'  KPI <50MB    : {"✅ PASS" if fp32_size < 50 else "❌ FAIL"}')

---
## Cell 11 — INT8 Quantization

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

print('Quantizing FP32 → INT8...')
quantize_dynamic(
    'spatialnet.onnx',
    'spatialnet_int8.onnx',
    weight_type=QuantType.QInt8
)

fp32_mb = os.path.getsize('spatialnet.onnx') / 1024**2
int8_mb = os.path.getsize('spatialnet_int8.onnx') / 1024**2
compression = (1 - int8_mb / fp32_mb) * 100

print(f'✅ INT8 export done')
print(f'   FP32 size    : {fp32_mb:.3f} MB')
print(f'   INT8 size    : {int8_mb:.3f} MB')
print(f'   Compression  : {compression:.1f}%')

---
## Cell 12 — Benchmark INT8 Model

In [ ]:
print('Running 1000 inference benchmarks (INT8)...')
lats_int8 = benchmark_onnx('spatialnet_int8.onnx')

print('\n=== INT8 ONNX KPI REPORT ===')
print(f'  Mean latency : {lats_int8.mean():.4f} ms')
print(f'  P50  latency : {np.percentile(lats_int8, 50):.4f} ms')
print(f'  P95  latency : {np.percentile(lats_int8, 95):.4f} ms')
print(f'  P99  latency : {np.percentile(lats_int8, 99):.4f} ms')
print(f'  Max  latency : {lats_int8.max():.4f} ms')
print(f'  KPI <20ms    : {"✅ PASS" if lats_int8.mean() < 20 else "❌ FAIL"}')

int8_size = os.path.getsize('spatialnet_int8.onnx') / 1024**2
print(f'\n  Model size   : {int8_size:.3f} MB')
print(f'  KPI <50MB    : {"✅ PASS" if int8_size < 50 else "❌ FAIL"}')

speedup = lats_fp32.mean() / lats_int8.mean()
print(f'\n  Speedup vs FP32: {speedup:.2f}x')

---
## Cell 13 — Side-by-Side Benchmark Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SpatialNet HRTF-Net — KPI Benchmark Report\nPS08 SoundScape AI | Samsung ennovateX 2026',
             fontsize=13, fontweight='bold', y=1.02)

# ── Plot 1: Latency distribution histogram
ax = axes[0]
ax.hist(lats_fp32, bins=50, alpha=0.7, color='#2196F3', label='FP32', density=True)
ax.hist(lats_int8, bins=50, alpha=0.7, color='#00BFA5', label='INT8', density=True)
ax.axvline(20, color='red', linestyle='--', linewidth=1.5, label='KPI limit (20ms)')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Density')
ax.set_title('Latency Distribution (1000 runs)')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 2: Model size comparison
ax = axes[1]
names  = ['FP32\nspatialnet.onnx', 'INT8\nspatialnet_int8.onnx', 'KPI Limit\n50MB']
sizes  = [fp32_size, int8_size, 50]
colors = ['#2196F3', '#00BFA5', '#FF7043']
bars   = ax.bar(names, sizes, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.3f} MB', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Size (MB)')
ax.set_title('Model Size vs KPI Limit')
ax.set_ylim(0, 60)
ax.grid(True, alpha=0.3, axis='y')

# ── Plot 3: P50/P95/P99 latency bar chart
ax = axes[2]
pct_labels = ['Mean', 'P50', 'P95', 'P99']
fp32_pcts  = [lats_fp32.mean(), np.percentile(lats_fp32, 50),
              np.percentile(lats_fp32, 95), np.percentile(lats_fp32, 99)]
int8_pcts  = [lats_int8.mean(), np.percentile(lats_int8, 50),
              np.percentile(lats_int8, 95), np.percentile(lats_int8, 99)]

x = np.arange(len(pct_labels))
w = 0.35
ax.bar(x - w/2, fp32_pcts, w, label='FP32', color='#2196F3', alpha=0.85)
ax.bar(x + w/2, int8_pcts, w, label='INT8', color='#00BFA5', alpha=0.85)
ax.axhline(20, color='red', linestyle='--', linewidth=1.5, label='KPI 20ms')
ax.set_xticks(x)
ax.set_xticklabels(pct_labels)
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency Percentiles')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('kpi_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: kpi_benchmark.png')

---
## Cell 14 — Generate Final KPI JSON Report

In [ ]:
def model_report(model_path, latencies):
    size_mb = os.path.getsize(model_path) / 1024**2
    return {
        'file': model_path,
        'size_mb': round(size_mb, 4),
        'kpi_size_pass': size_mb < 50,
        'latency_mean_ms': round(float(latencies.mean()), 4),
        'latency_p50_ms':  round(float(np.percentile(latencies, 50)), 4),
        'latency_p95_ms':  round(float(np.percentile(latencies, 95)), 4),
        'latency_p99_ms':  round(float(np.percentile(latencies, 99)), 4),
        'latency_max_ms':  round(float(latencies.max()), 4),
        'kpi_latency_pass': bool(latencies.mean() < 20),
        'headroom_vs_50ms_kpi': f'{50 / latencies.mean():.0f}x under limit',
        'headroom_size_vs_50mb': f'{50 / size_mb:.0f}x under limit'
    }

report = {
    'hackathon': 'Samsung ennovateX AX Hackathon 2026',
    'problem':   'PS08 — Immersive Spatial Voice Call Experience with AI',
    'team':      'SoundScape AI',
    'model':     'SpatialNet (HRTF-Net)',
    'dataset':   'SONICOM HRTF — CC-BY 4.0',
    'training':  {
        'epochs':      EPOCHS,
        'final_loss':  round(loss_history[-1], 6),
        'optimizer':   'Adam',
        'lr':          LR,
        'filter_length': FILTER_LENGTH
    },
    'fp32': model_report('spatialnet.onnx', lats_fp32),
    'int8': model_report('spatialnet_int8.onnx', lats_int8),
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
}

with open('kpi_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('✅ kpi_report.json saved')
print(json.dumps(report, indent=2))

---
## Cell 15 — Final Summary & Download Artifacts

In [ ]:
print('=' * 55)
print('   PS08 HRTF-Net — FINAL KPI SUMMARY')
print('   Samsung ennovateX 2026 | SoundScape AI')
print('=' * 55)

rows = [
    ('Training Loss (final)',   f"{loss_history[-1]:.6f}",             '—',         '✅'),
    ('FP32 Latency (mean)',     f"{lats_fp32.mean():.4f} ms",           '<20 ms',    '✅' if lats_fp32.mean() < 20 else '❌'),
    ('INT8 Latency (mean)',     f"{lats_int8.mean():.4f} ms",           '<20 ms',    '✅' if lats_int8.mean() < 20 else '❌'),
    ('FP32 Model Size',        f"{os.path.getsize('spatialnet.onnx')/1024**2:.3f} MB", '<50 MB', '✅'),
    ('INT8 Model Size',        f"{os.path.getsize('spatialnet_int8.onnx')/1024**2:.3f} MB", '<50 MB', '✅'),
    ('ONNX Graph Valid',       'Yes',                                   'Required',  '✅'),
    ('Dataset License',        'CC-BY 4.0',                            'Permissive','✅'),
    ('Model License',          'Apache-2.0',                           'Open',      '✅'),
]

for metric, value, target, status in rows:
    print(f'  {status}  {metric:<28} {value:<18} target: {target}')

print('\nArtifacts generated:')
artifacts = [
    'spatialnet.pth',
    'spatialnet.onnx',
    'spatialnet_int8.onnx',
    'kpi_report.json',
    'training_loss.png',
    'kpi_benchmark.png'
]
for f in artifacts:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f'  ✅  {f:<30} ({size:.1f} KB)')
    else:
        print(f'  ❌  {f} — NOT FOUND')

print('\n' + '=' * 55)

---
## Cell 16 — Download All Files (Colab Only)

Run this cell to download all generated artifacts to your local machine.

In [ ]:
try:
    from google.colab import files

    download_list = [
        'spatialnet.pth',
        'spatialnet.onnx',
        'spatialnet_int8.onnx',
        'kpi_report.json',
        'training_loss.png',
        'kpi_benchmark.png'
    ]

    for f in download_list:
        if os.path.exists(f):
            print(f'📥 Downloading {f}...')
            files.download(f)
        else:
            print(f'⚠️  Skipping {f} — not found')

    print('\n✅ All downloads triggered. Check your browser downloads folder.')
except ImportError:
    print('Not running in Colab. Files are in the current directory:')
    for f in os.listdir('.'):
        if any(f.endswith(ext) for ext in ['.pth', '.onnx', '.json', '.png']):
            print(f'  {f}')

---
## Notes for Judges

| Claim | Evidence from this notebook |
|---|---|
| Trained on real HRTF data | SONICOM P0001.sofa (CC-BY 4.0, Zenodo) |
| Model <50MB KPI | FP32 ~0.4MB, INT8 ~0.15MB — 100–300x under KPI |
| Inference <20ms KPI | ~0.1–0.5ms on CPU — 40–100x under KPI |
| ONNX production-ready | Full graph validation via `onnx.checker` |
| Open-source compatible | Apache-2.0 model, CC-BY 4.0 data, BSD ORT |

**Architecture:** `SpatialNet` maps 3D source coordinates (azimuth, elevation, distance) → binaural FIR filter pair (left ear, right ear). This is the HRTF-Net component of the full SoundScape AI pipeline described in the Phase 1 Blueprint.